In [0]:
from pyspark.sql.functions import current_timestamp, col
from dataclasses import dataclass

@dataclass
class AutoLoaderConfig:
    source_path: str
    checkpoint_path: str
    table_name: str
    file_format: str = "csv"
    header: bool = True
    infer_column_types: bool = True

class BronzeAutoLoader:
    def __init__(self, config: AutoLoaderConfig):
        self.config = config
        self.spark = spark

    def _build_reader(self):
        return (
            self.spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", self.config.file_format)
            .option("header", str(self.config.header).lower())
            .option("cloudFiles.schemaLocation", self.config.checkpoint_path)
            .option("cloudFiles.inferColumnTypes", str(self.config.infer_column_types).lower())
            .option("cloudFiles.schemaHints", "transaction_timestamp string")
            .load(self.config.source_path)
        )

    def _enrich(self, raw_df):
        return raw_df.withColumns({
            "_ingested_at": current_timestamp(),
            "_source_file": col("_metadata.file_path")
        })

    def run(self):
        try:
            raw_df = self._build_reader()
            bronze_df = self._enrich(raw_df)
            query = (
                bronze_df.writeStream
                .format("delta")
                .option("checkpointLocation", f"{self.config.checkpoint_path}/_delta_checkpoint")
                .option("mergeSchema", "true")
                .trigger(availableNow=True)
                .toTable(self.config.table_name)
            )
            query.awaitTermination()
            return query
        except Exception as e:
            raise RuntimeError(f"BronzeAutoLoader failed to write to table '{self.config.table_name}': {e}") from e

config = AutoLoaderConfig(
    source_path="/Volumes/retail_q/volumes/blob_source/transactions_source/",
    checkpoint_path="/Volumes/retail_q/volumes/blob_source/transactions_source/_checkpoints/transaction",
    table_name="retail_q.blob_bronze.transaction",
)

loader = BronzeAutoLoader(config)
loader.run()

In [0]:
transactions_df = spark.sql(f"""
SELECT * FROM 
retail_q.blob_bronze.transaction                      
""")
print(f"The Total records in the transaction table is {transactions_df.count()}")
display(transactions_df)